# Schizophrenia Pathway Classifier — EDA

## Objective
In this notebook I will load the `GSE21138` dataset and inspect it, by first checking data quality and class balance to ensure the data is prepared enough to do accurate analysis and determine any potential feature engineering decisions. Then, after ensuring the data is good, conduct the actual exploratory analysis — seeing if the expression data separates by diagnosis or duration in reduced dimensions. Finally, concluding with a summary of EDA findings.

## Inputs
Dataset: `GSE21138` from "Gene Expression Profiles in BA46 of Subjects with Schizophrenia and Matched Controls".

Contains 30 subjects with schizophrenia and 29 age- and sex-matched controls, Ages (18-81 years), and three duration categories (short doi=<5 yrs; intermediate doi=7-18yrs; long doi=>28 yrs). 'Cont-7' was determined to be an outlier, and was removed from the publication analysis. 

Platform: GPL570 / Affymetrix Human Genome U133 Plus 2.0 Array

## 1.1 Setup & Imports

Import necessary libraries for data manipulation and plotting: numpy, pandas, matplotlib, seaborn, GEOparse. Additionally, set theme and figure size for plots. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse 

sns.set_theme()
plt.rcParams['figure.figsize'] = (10, 6)

## 1.2 Data Acquisition 
To access the GEO dataset I use `GEOparse.get_GEO` to save the GEO in `../data/raw`, and then confirm data was loaded properly by checking object type and sample count. 

In [2]:
gse = GEOparse.get_GEO(geo='GSE21138', destdir="../data/raw")

17-Aug-2026 13:11:59 DEBUG utils - Directory ../data/raw already exists. Skipping.
17-Aug-2026 13:11:59 INFO GEOparse - File already exist: using local version.
17-Aug-2026 13:11:59 INFO GEOparse - Parsing ../data/raw/GSE21138_family.soft.gz: 
17-Aug-2026 13:11:59 DEBUG GEOparse - DATABASE: GeoMiame
17-Aug-2026 13:11:59 DEBUG GEOparse - SERIES: GSE21138
17-Aug-2026 13:11:59 DEBUG GEOparse - PLATFORM: GPL570
/Users/joshuasim/Desktop/summer_projects/schizophrenia-pathway-classifier/venv/lib/python3.12/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
17-Aug-2026 13:12:00 DEBUG GEOparse - SAMPLE: GSM528831
17-Aug-2026 13:12:00 DEBUG GEOparse - SAMPLE: GSM528832
17-Aug-2026 13:12:01 DEBUG GEOparse - SAMPLE: GSM528833
17-Aug-2026 13:12:01 DEBUG GEOparse - SAMPLE: GSM528834
17-Aug-2026 13:12:01 DEBUG GEOparse - SAMPLE: GSM528835
17-Aug

In [33]:
print(type(gse))
len(gse.metadata['sample_id'])

<class 'GEOparse.GEOTypes.GSE'>


59

`.type()` returned `<class 'GEOparse.GEOTypes.GSE'>` confirming the correct object type, and `len(gse.metadata['sample_id'])` returned 59 confirming all samples were loaded. Further checks will be done to ensure samples' contents are correct. 

## 1.3 Sample Metadata Extraction & Validation

I check `GSM528831`'s, the first sample in the dataset, metadata to see where to extract information, then create a dataframe containing all samples with their diagnosis and illness duration to be used downstream (checking class balance, group comparisons, etc.). 

In [5]:
gse.gsms['GSM528831'].metadata

{'title': ['Control-1'],
 'geo_accession': ['GSM528831'],
 'status': ['Public on Mar 31 2010'],
 'submission_date': ['Mar 30 2010'],
 'last_update_date': ['Aug 12 2022'],
 'type': ['RNA'],
 'channel_count': ['1'],
 'source_name_ch1': ['prefrontal cortex'],
 'organism_ch1': ['Homo sapiens'],
 'taxid_ch1': ['9606'],
 'characteristics_ch1': ['brain region: BA46',
  'stage of illness [short doi=<5 yrs; intermediate doi=7-18yrs; long doi=>28 yrs]: short duration of illness - control',
  'Sex: M',
  'age: 38',
  'tissue ph: 6.42',
  'pmi (hrs): 46',
  'type of drug: NA',
  'drug dose (chlorpromazine equivalents): NA'],
 'molecule_ch1': ['total RNA'],
 'extract_protocol_ch1': ['Total RNA was extracted from the prefrontal cortex (Brodmann Area 46; 100 mg; left hemisphere) from all subjects as described previously (Desplats et al., 2006).  RNA quantification was determined by spectrophotometer readings, and quality by Agilent Bioanalyzer scans.  RNA integrity (RIN) numbers were not available at

Looking at `GSM528831`'s metadata, `characteristics_ch1` is the key that contains the information on diagnosis and duration of illness.

I will create a dataframe using `characteristics_ch1` to extract duration and diagnosis. 

I will do this by looping over all the samples (`gse.gsms.items()`) applying the following parsing logic to each sample to create a dictionary of sample : sampleID/duration/diagnosis: splitting `characteristics_ch1`'s second item (contains the diagnosis and duration), on `:` to get two strings - one containing the duration and diagnosis, the other containing contextual information. I will then split that list again on the second item by `-` to get a new list containing only duration information and diagnosis. Lastly, I set `diagnosis` to the split string's second item which has only the diagnosis, and set `duration` to the first item of that split, after stripping whitespace and splitting on ' ' to isolate just the duration. 

I put that dictionary into a dataframe and apply `.T` to transpose it so that each row is a sample and the columns are `['sample_id', 'duration', 'diagnosis']`. 

In [3]:
df = {}
for gsm_id, gsm_obj in gse.gsms.items():
   a = gsm_obj.metadata['characteristics_ch1'][1].split(':')
   b = a[1].split('-')
   duration = b[0].strip().split(' ')[0]
   diagnosis = b[1].strip()
   sample_id = gsm_id
   df[gsm_id] = [sample_id, duration, diagnosis]
   
gsm_df = pd.DataFrame(df).T
gsm_df.columns = ['sample_id', 'duration', 'diagnosis']
gsm_df.head()

,sample_id,duration,diagnosis
GSM528831,GSM528831,short,control
GSM528832,GSM528832,short,control
GSM528833,GSM528833,short,control
GSM528834,GSM528834,short,control
GSM528835,GSM528835,short,control


`.head()` of `gsm_df` confirms the dataframe has correactly labeld columns and rows are samples.

To further check the dataframe was constructed correctly, I will check a specific schizophrenia sample to ensure that those samples were loaded correctly, and the `.value_counts()` of `diagnosis` and `duration` to ensure the correct number of diagnoses and three duration categories are present. 

In [25]:
gsm_df.loc['GSM528885']

sample_id        GSM528885
duration              long
diagnosis    schizophrenia
Name: GSM528885, dtype: str

`GSM528885` shows `duration = long` and `diagnosis = schizophrenia` — confirming schizophrenia samples were parsed correctly.

In [24]:
print(gsm_df['diagnosis'].value_counts())
gsm_df['duration'].value_counts()

diagnosis
schizophrenia    30
control          29
Name: count, dtype: int64


duration
intermediate    28
long            16
short           15
Name: count, dtype: int64

`value_counts()` for both diagnosis and duration show the correct number of diagnoses (30 schizophrenia and 29 control — cont 7 was considered an outlier and excluded) and the three duration categories (intermediate: 28, long: 16, short: 15).

I will run a `pd.crosstab` on `diagnosis` and `duration` to check the spread of control and schizophrenia durations. 

In [26]:
pd.crosstab(gsm_df['diagnosis'], gsm_df['duration'])

duration,intermediate,long,short
diagnosis,,,
control,14,8,7
schizophrenia,14,8,8


The crosstab shows a proportional pattern (control 14/8/7 vs. schizophrenia 14/8/8) that is consistent with control being stage matched to schizophrenia subjects, even though the original study summary only explicitly states age/sex-matching. 

## 1.4 Load/Inspect Expression Matrix

To load the epxression matrix, I identify the correct attribute that has samples and their actual expression values in order to use it to construct the expression matrix. Then I perform sanity checks on the constructed matrix to ensure it can be used downstream for...

I use `dir(gse)` to look at the different attributes/methods in `gse` in order to identify which attribute/method might hold the expression values, so that it can be used to construct the expression matrix. 

In [4]:
dir(gse)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__metaclass__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_get_metadata_as_string',
 '_get_object_as_soft',
 '_phenotype_data',
 'database',
 'download_SRA',
 'download_supplementary_files',
 'geotype',
 'get_accession',
 'get_metadata_attribute',
 'get_type',
 'gpls',
 'gsms',
 'merge_and_average',
 'metadata',
 'name',
 'phenotype_data',
 'pivot_and_annotate',
 'pivot_samples',
 'relations',
 'show_metadata',
 'to_soft']

`pivot_and_annotate` and `pivot_samples` seem to be the most appropriate as their names suggest having sample or annotated data.

I look at `GSM528831` `.table` to see what columns it stores to determine what columns a `gse` attribute needs to have in order to construct the expression matrix. 

In [19]:
gse.gsms['GSM528831'].table.head()

,ID_REF,VALUE
0,AFFX-BioC-5_at,8.7921
1,AFFX-BioC-3_at,9.1502
2,AFFX-BioDn-5_at,10.2670
3,AFFX-DapX-5_at,8.7025
4,AFFX-DapX-M_at,9.5005


This shows that the `gsms` has `ID_REF` and `VALUE` stored, so an attribute that takes `ID_REF` and `VALUE` is needed to construct the matrix. 

I use `help()` to see if the arguments the two attributes take match the `ID_REF` and `VALUE` columns for samples.

In [23]:
print(help(gse.pivot_and_annotate))
help(gse.pivot_samples)

Help on method pivot_and_annotate in module GEOparse.GEOTypes:

pivot_and_annotate(values, gpl, annotation_column, gpl_on='ID', gsm_on='ID_REF') method of GEOparse.GEOTypes.GSE instance
    Annotate GSM with provided GPL.

    Args:
        values (:obj:`str`): Column to use as values eg. "VALUES"
        gpl (:obj:`pandas.DataFrame` or :obj:`GEOparse.GPL`): A d or
            DataFrame to annotate with.
        annotation_column (:obj:`str`): Column in table for annotation.
        gpl_on (:obj:`str`, optional): Use this column in GPL to merge.
            Defaults to "ID".
        gsm_on (:obj:`str`, optional): Use this column in GSM to merge.
            Defaults to "ID_REF".

    Returns:
        pandas.DataFrame: Pivoted and annotated table of results

None
Help on method pivot_samples in module GEOparse.GEOTypes:

pivot_samples(values, index='ID_REF') method of GEOparse.GEOTypes.GSE instance
    Pivot samples by specified column.

    Construct a table in which columns (names) ar

While both take `ID_REF` as an index and `VALUE` as an argument, `pivot_and_annotate` also takes an additional column for annotation, so I will use `pivot_samples` to construct the matrix since it is simpiler and only requires the two arguments `ID_REF` and `VALUE`. 

I load the expression matrix using `.pivot_samples` with `values='VALUE'`, and then look at `.head()` to check values look real. 

In [25]:
exp_matrix = gse.pivot_samples(values='VALUE')
exp_matrix.head()

name,GSM528831,GSM528832,GSM528833,GSM528834,GSM528835,GSM528836,GSM528837,GSM528838,GSM528839,GSM528840,...,GSM528880,GSM528881,GSM528882,GSM528883,GSM528884,GSM528885,GSM528886,GSM528887,GSM528888,GSM528889
ID_REF,,,,,,,,,,,,,,,,,,,,,
1007_s_at,10.0887,9.1448,9.5847,9.2438,9.0333,9.3003,9.2000,9.3305,9.0366,9.6668,...,9.4743,10.1502,9.2517,9.0640,9.5347,9.7102,9.6754,10.0982,9.8905,10.4207
1053_at,7.0144,7.0536,6.9642,7.2024,6.6731,6.9244,6.9524,6.9647,6.6808,7.0324,...,6.1323,6.6461,6.7076,6.8556,7.1541,7.2579,6.9427,6.7352,6.8509,7.1291
117_at,4.8340,5.1611,4.9111,5.2503,4.8153,4.5898,4.2863,4.8058,4.8646,5.0324,...,8.1406,5.4431,4.6838,5.0441,5.3910,4.1443,4.8502,4.8425,4.0438,5.2339
121_at,7.9776,8.5253,8.1324,7.6403,7.4291,8.1697,7.1497,7.9566,8.2069,7.9247,...,7.6873,7.9584,7.5622,7.9827,7.7387,7.5936,7.8277,7.6981,7.7678,7.1596
1255_g_at,5.3040,5.8394,5.3994,5.5726,5.3081,5.4553,5.1663,5.7873,6.0013,5.1592,...,5.4498,4.9400,4.6725,5.5831,5.0636,5.5238,5.2938,5.1384,4.5343,4.7568


I perform sanity checks on the expression matrix using `.shape` to ensure rows/columns are correct, and check if `exp_matrix` samples match the `gsm_df` samples. 

In [26]:
print(exp_matrix.shape)
exp_matrix.columns == gsm_df['sample_id']

(30061, 59)


GSM528831    True
GSM528832    True
GSM528833    True
GSM528834    True
GSM528835    True
GSM528836    True
GSM528837    True
GSM528838    True
GSM528839    True
GSM528840    True
GSM528841    True
GSM528842    True
GSM528843    True
GSM528844    True
GSM528845    True
GSM528846    True
GSM528847    True
GSM528848    True
GSM528849    True
GSM528850    True
GSM528851    True
GSM528852    True
GSM528853    True
GSM528854    True
GSM528855    True
GSM528856    True
GSM528857    True
GSM528858    True
GSM528859    True
GSM528860    True
GSM528861    True
GSM528862    True
GSM528863    True
GSM528864    True
GSM528865    True
GSM528866    True
GSM528867    True
GSM528868    True
GSM528869    True
GSM528870    True
GSM528871    True
GSM528872    True
GSM528873    True
GSM528874    True
GSM528875    True
GSM528876    True
GSM528877    True
GSM528878    True
GSM528879    True
GSM528880    True
GSM528881    True
GSM528882    True
GSM528883    True
GSM528884    True
GSM528885    True
GSM528886 

In [29]:
exp_matrix.describe()

name,GSM528831,GSM528832,GSM528833,GSM528834,GSM528835,GSM528836,GSM528837,GSM528838,GSM528839,GSM528840,...,GSM528880,GSM528881,GSM528882,GSM528883,GSM528884,GSM528885,GSM528886,GSM528887,GSM528888,GSM528889
count,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,...,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000,30061.000000
mean,6.096415,6.096083,6.084213,5.989972,5.996463,6.043386,6.009915,6.051067,6.087311,6.055251,...,6.157749,6.176198,5.882081,6.074273,6.036868,6.027844,6.066820,6.197870,6.073197,6.084032
std,1.609117,1.565847,1.579749,1.590782,1.605292,1.571401,1.601515,1.556544,1.549775,1.593954,...,1.634704,1.639886,1.664364,1.549392,1.593269,1.608989,1.606358,1.608165,1.589823,1.677716
min,2.077700,2.115400,2.156900,2.165800,2.165800,1.973500,2.114500,2.208000,2.158900,2.114500,...,2.056600,2.032400,1.894600,2.144300,2.109300,2.165400,2.222600,2.217500,2.165400,2.160900
25%,4.914300,4.943200,4.916100,4.821800,4.822300,4.896700,4.830800,4.899200,4.944700,4.862200,...,4.965500,4.955900,4.689600,4.929800,4.856500,4.837500,4.883900,5.010400,4.904800,4.835700
50%,5.807000,5.836600,5.798900,5.691300,5.700600,5.771500,5.710600,5.772500,5.816800,5.761200,...,5.906300,5.892100,5.614500,5.787200,5.744600,5.730400,5.753800,5.909600,5.774200,5.776500
75%,7.061100,7.033600,7.010800,6.941200,6.961600,6.943700,6.962600,6.959200,6.978200,7.019700,...,7.161800,7.164000,6.846500,6.982300,6.996400,6.988900,7.006500,7.152500,7.013800,7.140400
max,12.752700,12.778100,12.805800,12.818400,12.834500,12.703500,12.769400,12.757200,12.783900,12.770600,...,13.354800,12.789800,12.651200,12.754900,12.825200,33.794900,34.940900,32.309000,12.745600,34.840700


`.shape` confirms the correct number of samples were included (59), but the row count 30061 suggests something got filtered/collapsed upstream somewhere since GPL570 platform normally has 54,675 probesets. 

The samples in exp_matrix match the gsm_df samples

The 

## 1.5 Data Quality Check (sanity cjeckz)

## 1.6 Class Balances

## 1.7 Dimensionality Reduction — PCA / Clustering

## 1.8 EDA Summary 